<a href="https://colab.research.google.com/github/adidror005/youtube-videos/blob/main/Meta_Muse_Latest_Stock_News_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latest Stock News with Meta Muse + Web Search

A short, **Google Colab-ready** example using Meta's **Responses API** and built-in **`web_search`** tool.

**What it does:** For each ticker, Muse searches for recent news and produces a source-backed summary with possible market implications. The notebook also extracts the *actual search results* and *citation annotations* into pandas DataFrames you can export.

**Before running:** Create a Meta Model API key from [Meta's developer platform](https://dev.meta.ai/) and add it to **Colab → Secrets (🔑) → `MODEL_API_KEY`**. Enable notebook access to the secret.

**Important:** This is **research, not a breaking-news feed or trading signal**. Search may not happen on every request, results may be incomplete or stale, timestamps may be absent, and a cited source does not prove a headline occurred within the requested window. Verify articles and publication times before making trading decisions.

[Official search grounding documentation](https://dev.meta.ai/docs/search-grounding) · [Meta API cookbook](https://dev.meta.ai/docs/cookbook/search-grounding)

## 1. Install the Python libraries

%pip -q install -U openai pandas

## 2. Connect to Meta's API

This uses the standard **OpenAI Python SDK** with Meta's API URL. Your key is read from Colab Secrets or the `MODEL_API_KEY` environment variable, **never hardcoded**.

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("MODEL_API_KEY")
except (ImportError, Exception):
    api_key = None

api_key = api_key or os.environ.get("MODEL_API_KEY")
if not api_key:
    raise RuntimeError(
        "Add MODEL_API_KEY to Colab Secrets (key icon in left sidebar) "
        "and enable access, or set the MODEL_API_KEY environment variable."
    )

client = OpenAI(
    base_url="https://api.meta.ai/v1",
    api_key=api_key,
    timeout=90.0,
)
MODEL = model="muse-spark-1.3-contributor"
print("Meta API client ready.")

Meta API client ready.


## 3. Choose stocks and search window

Edit the tickers or time window below. This notebook sends **one request per ticker**; requests and underlying search queries may incur charges.

In [ ]:
TICKERS = "META, NVDA, AMD"   # e.g. "AAPL, MSFT, GOOGL"
HOURS_BACK = 48               # search the last 48 hours
MAX_STORIES = 5               # maximum news items requested per ticker
SEARCH_CONTEXT = "medium"    # "low", "medium", or "high"

import re
from datetime import datetime, timedelta, timezone

tickers = [t.strip().upper() for t in TICKERS.split(",") if t.strip()]
tickers = list(dict.fromkeys(tickers))  # remove duplicates, preserve order
if not tickers or any(not re.fullmatch(r"[A-Z0-9.\^-]{1,12}", t) for t in tickers):
    raise ValueError("Use a comma-separated list of standard ticker symbols.")
if HOURS_BACK <= 0 or MAX_STORIES <= 0:
    raise ValueError("HOURS_BACK and MAX_STORIES must be positive.")

checked_at = datetime.now(timezone.utc)
window_start = checked_at - timedelta(hours=HOURS_BACK)
print(f"Tickers: {', '.join(tickers)}")
print(f"Window (UTC): {window_start:%Y-%m-%d %H:%M} to {checked_at:%Y-%m-%d %H:%M}")

Tickers: META, NVDA, AMD
Window (UTC): 2026-11-15 09:29 to 2026-11-17 09:29


## 4. Search and extract sources

`web_search_call.results` holds **pages the model retrieved**. `url_citation` annotations hold the **subset the answer cited**, including which text passage was supported. They are different and we extract both. The model decides whether to search; we verify it did.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display


def search_stock_news(ticker, now_utc, start_utc):
    prompt = f"""
    Current UTC time: {now_utc:%Y-%m-%d %H:%M}.
    Research the latest material stock-market/company news for {ticker},
    preferably published between {start_utc:%Y-%m-%d %H:%M} UTC and
    {now_utc:%Y-%m-%d %H:%M} UTC. Use live web search.

    Give up to {MAX_STORIES} distinct recent developments, newest first.
    For each one include:
    - A short, specific headline and the publisher/source
    - Publication date and time (with timezone) **only if verified**
    - A 1-2 sentence factual summary with citations
    - Possible bullish, bearish, or mixed implications **as analysis,
      not established fact**

    Explicitly distinguish event dates from article publication dates.
    Do not present old articles as recent. If fewer than {MAX_STORIES}
    suitable stories are found, return fewer. If none can be verified,
    clearly say so. Do not invent headlines, prices, timestamps, or URLs.
    """
    return client.responses.create(
        model=MODEL,
        input=prompt,
        tools=[{"type": "web_search", "search_context_size": SEARCH_CONTEXT}],
        include=["web_search_call.results"],
    )


def unpack_news_response(ticker, response, checked_at):
    # Convert SDK objects to plain dictionaries; handles optional fields safely.
    payload = response.model_dump(mode="json")
    items = payload.get("output") or []
    search_calls = [i for i in items if i.get("type") == "web_search_call"]
    completed_search = any(i.get("status") == "completed" for i in search_calls)

    raw_hits, citations, answers = [], [], []
    for item in search_calls:
        for result in item.get("results") or []:
            if result.get("url"):
                raw_hits.append({
                    "ticker": ticker,
                    "checked_utc": checked_at.isoformat(),
                    "title": result.get("title") or "",
                    "url": result["url"],
                    "snippet": result.get("snippet") or "",
                })

    for item in items:
        if item.get("type") != "message":
            continue
        for block in item.get("content") or []:
            if block.get("type") != "output_text":
                continue
            block_text = block.get("text") or ""
            if block_text:
                answers.append(block_text)
            for annotation in block.get("annotations") or []:
                if annotation.get("type") != "url_citation" or not annotation.get("url"):
                    continue
                start, end = annotation.get("start_index"), annotation.get("end_index")
                supported = ""
                if isinstance(start, int) and isinstance(end, int) and 0 <= start <= end <= len(block_text):
                    supported = block_text[start:end]
                citations.append({
                    "ticker": ticker,
                    "checked_utc": checked_at.isoformat(),
                    "title": annotation.get("title") or "",
                    "url": annotation["url"],
                    "supported_passage": supported,
                })

    return {
        "searched": completed_search,
        "num_search_calls": len(search_calls),
        "text": "\n\n".join(answers),
        "raw_hits": raw_hits,
        "citations": citations,
    }

## 5. Run the stock-news scanner

You get a report for each ticker **only if the API confirms a completed search and includes citations**. Otherwise the notebook explicitly flags it instead of silently passing off an ungrounded answer as recent news.

In [ ]:
all_results = {}
source_rows, citation_rows, summary_rows = [], [], []

for ticker in tickers:
    display(Markdown(f"## {ticker}"))
    try:
        response = search_stock_news(ticker, checked_at, window_start)
        result = unpack_news_response(ticker, response, checked_at)
    except Exception as exc:
        print(f"API error for {ticker}: {type(exc).__name__}: {exc}")
        summary_rows.append({"ticker": ticker, "status": "API error", "answer": ""})
        continue

    all_results[ticker] = result
    source_rows.extend(result["raw_hits"])
    citation_rows.extend(result["citations"])

    if not result["searched"]:
        print("⚠️ No completed web search detected. Not treating this as current news.")
        status = "No completed search"
    elif not result["citations"]:
        print("⚠️ Search ran but no citations were returned. Not treating this as sourced news.")
        status = "No citations"
    else:
        status = "Grounded response (verify article timestamps)"
        display(Markdown(result["text"] or "*No summary returned.*"))

        # Deduplicate cited links so the user can inspect the underlying articles.
        seen = set()
        unique = []
        for c in result["citations"]:
            if c["url"] not in seen:
                seen.add(c["url"])
                unique.append(c)
        display(Markdown("**Cited sources:**\n" + "\n".join(
            f"- [{c['title'] or c['url']}]({c['url']})" for c in unique
        )))

    print(f"Search calls: {result['num_search_calls']} | "
          f"raw pages: {len(result['raw_hits'])} | "
          f"citation annotations: {len(result['citations'])}")
    summary_rows.append({
        "ticker": ticker,
        "status": status,
        "answer": result["text"] if status.startswith("Grounded") else "",
    })

## META

I'll search live for fresh META developments from the last 48 hours.

I've spotted several candidates — now I'll verify their dates and details.

Four stories confirmed so far — I'll look for one more development to round out the set.

Live web search for **META / Meta Platforms** covering publication window **2026-11-15 09:29 UTC to 2026-11-17 09:29 UTC**.

Found **5 distinct developments**, newest first by **verified article publication date**, not event date.

### 1. Meta-linked AI data-center debt held by insurers under scrutiny
**Headline:** “Meta Platforms (META) Faces Fresh Questions As Insurers Pile Into AI Debt” — **Simply Wall St News**
**Published:** 2026-11-17T02:09:53.834Z — verified as UTC indicated by “Z”
**Event vs article:** Article publication is Nov. 17; underlying investigation into insurer holdings is described as “recent” without a separate dated event given.

**Summary:** Meta is linked to a growing pool of insurer investments in AI data-center debt, with large life insurers reported buying complex securities tied to Meta-backed infrastructure where Meta often holds only partial ownership. The structures reportedly rely on Meta guarantees and strong ratings while leaving full risk difficult to assess, against flagged 2026 AI capex of US$130-145B and multi-year compute commitments.

**Implication — analysis, not fact:** **Mixed to bearish:** highlights off-balance-sheet-style funding reliance and capital-intensity risk; **bullish angle:** shows third-party capital appetite for Meta-anchored AI assets.

### 2. Plaintiffs seek sanctions in youth-harm MDL over privilege claims
**Headline:** “Plaintiffs Seek Sanctions Against Meta for Withholding 93% of Review Files” — **KuCoin, citing ChainCatcher report / Reuters**
**Published:** 2026-11-17T01:05:31.000Z
**Event vs article:** Article page published Nov. 17 UTC; it cites Reuters reporting dated November 16 and plaintiffs’ filing made “on Friday” — i.e., filing event predates article.

**Summary:** Lawyers for thousands of individuals and school districts asked the judge to sanction Meta, alleging platform design exacerbated youth mental-health crisis and that privilege was used to conceal relevant evidence. Plaintiffs say they received tens of thousands of new documents only three months before the next hearing despite disclosure deadlines 1.5 years ago, and ask Judge Yvonne Gonzalez Rogers to instruct the February-trial jury about Meta’s conduct and limit Meta’s challenges. Meta says it acted in good faith and calls the motion litigation strategy; plaintiffs claim ~93% were improperly redacted/withheld with review of 20,000 documents/week ongoing in ~4,300 consolidated cases MDL No. 4:22-md-03047.

**Implication — analysis:** **Bearish:** raises litigation-overhang and discovery-risk perception ahead of February trial; not a liability finding.

### 3. Shares fell 2.4% on Monday, Nov. 16 to $626.41
**Headline:** “Meta Platforms (NASDAQ:META) Stock Price Down 2.4% – Here’s Why” — **themarketsdaily.com, posted by Danessa Lincoln**
**Published:** Nov 16th, 2026 — date verified, time not shown
**Event vs article:** Market event is Monday trading; article is same-day recap posted Nov. 16.

**Summary:** Shares traded down 2.4% Monday, trading as low as $620.14 and last at $626.41, after previously closing at $641.69. The recap links the move to Meta’s stated plan to appeal Italy’s €805 million (about $887 million) fine and to November issuance of $15 billion in dollar bonds plus €9 billion in euro bonds alongside Q3 capex of $34.9 billion and negative free cash flow.

**Implication — analysis:** **Bearish near-term price action:** reflects combined regulatory + leverage + AI-spending concerns; **bullish contrarian angle:** dip could be viewed as entry if cash-flow recovers — opinion only.

### 4. Italy’s €805M fine over Android gallery suggestions — compliance order
**Headline:** “Meta Platforms stock: Italy fine, 574.60 euros, up 0.31%” — **ad-hoc-news.de**
**Published:** 11/16/2026 at 07:53 per byline; timezone for byline not specified
**Event vs article:** Regulatory event was Saturday, November 14; article is Nov. 16 follow-up. Italian authorities fined Meta 805 million euros on November 14 over Android photo-gallery access without specific consent.

**Summary:** Penalty concerns Facebook’s camera-roll suggestions recommending Android gallery photos/videos for posts, found to rely on a single app-wide permission without sufficiently specific information, violating Italian implementation of the EU ePrivacy directive. Meta must seek separate permission, explain and allow disablement, and keep it off by default within 60 days, with right to appeal. Frankfurt context: Xetra closed at EUR 572.80 Friday, Nov. 13; Lang & Schwarz traded at 574.60 euros at 07:52 CET Monday, Nov. 16, up 0.31%.

**Implication — analysis:** **Bearish:** adds compliance cost and EU regulatory risk; **mixed:** fine size is small relative to scale, and appeal/60-day remedy leaves room for containment — opinion only.

### 5. Breakdown of ~$25B November borrowing and cash burn
**Headline:** “Is Meta Stock Still a Buy After Borrowing About $25 Billion This Month?” — **The Motley Fool**
**Published:** 2026-11-15T23:47:01.000Z
**Event vs article:** Analysis published Nov. 15 UTC; borrowing events were earlier — $15B dollar bonds Nov. 3 and €9B euro bonds (~$10B) Nov. 4, its first euro deal, plus late-October stock sale at $625 plus mandatory convertible preferred.

**Summary:** Author notes ~$25B borrowed this month with stock near $640, ~18% below Sept. 24 closing high ~$778. Capex including finance-lease principal rose $19.8B in Q1 to $31.1B in Q2 to $34.9B in Q3 while operating cash flow stayed $32-33B, pushing free cash flow from $12.4B in Q1 to $784M in Q2 to negative $2.1B in Q3. With 2026 capex guidance $135-145B and ~$85.8B spent through September, Q4 implies another $49-59B; new dollar notes pay 5.85% due 2031 to 7.5% due 2056, above any pre-2026 Meta bond high of 5.75%, euro notes 4.3% to 6.4%, Q3 interest expense quadrupled to $977M with ~$1.5B added annually, versus Q3 ad revenue +26% to $63.1B and Q4 revenue guide $73-76B.

**Implication — analysis:** **Mixed:** author argues debt is affordable given $22.9B Q3 operating income ex $9.89B legal charge and investment-grade ratings, but flags Q4 as potentially most expensive quarter and risk of more 6%-7.5% borrowing — opinion only.


**Cited sources:**
- [Meta Platforms (META) Faces Fresh Questions As Insurers Pile Into AI Debt](https://simplywall.st/stocks/us/media/nasdaq-meta/meta-platforms/news/meta-platforms-meta-faces-fresh-questions-as-insurers-pile-i)
- [Plaintiffs Seek Sanctions Against Meta for Withholding 93% of Review Files](https://www.kucoin.com/news/flash/plaintiffs-seek-sanctions-against-meta-for-withholding-93-of-review-files)
- [Meta Platforms (NASDAQ:META) Stock Price Down 2.4% – Here’s Why](https://www.themarketsdaily.com/2026/11/16/meta-platforms-nasdaqmeta-stock-price-down-2-4-heres-why.html)
- [Meta Platforms stock: Italy fine, 574.60 euros, up 0.31%](https://www.ad-hoc-news.de/boerse/vorboerse/meta-platforms-stock-italy-fine-574-60-euros-up-0-31/70519537)

Search calls: 13 | raw pages: 83 | citation annotations: 25


## NVDA

I'll research the latest NVDA developments from the past 48 hours, starting with live news searches.

Initial results point to several time-sensitive stories. I'll open the full articles to verify publication dates and details.

I've confirmed three developments with verified timestamps. I'll check a couple more sources to round out the coverage.

Live web search for `NVDA` news in the requested window — **2026-11-15 09:29 UTC to 2026-11-17 09:29 UTC**.

Found 5 distinct items with verified publication dates. Ordered newest first by **article publication time**, not event date.

### 1. Analyst cuts into earnings — selloff deepens
**Headline:** “NVIDIA (NVDA) Stock Price Prediction: Selloff deepens as earnings, leadership shock and analyst cuts collide” — **Blockchain.News**
**Publication (verified):** Nov 17, 2026 09:16 UTC

**What happened vs when published:**
- Article publication: Nov 17 09:16 UTC, 13 minutes before the 09:29 UTC cutoff.
- Events described: Nov 16 close / Nov 16 analyst actions / Nov 17 earnings (upcoming).

**Factual summary:** NVDA closed at $156.14 on November 16, down 4.12%. Heading into its November 17 earnings report, CNBC reported on November 16 that two more firms trimmed targets, with Mizuho cutting to $220 from $250 and UBS to $240 from $300, while TradingView also reported Morgan Stanley lowered to $267 from $300 while keeping Overweight.

**Analysis (not fact):** Bearish near-term — cuts raise the risk premium and signal reduced conviction; mixed to bullish if oversold positioning unwinds on a beat.

### 2. Treasury Secretary scrutiny claim + securities class-action solicitation
**Headline:** “NVIDIA Corporation (NVDA): Treasury Secretary Reportedly …” — **Hagens Berman via GlobeNewswire**
**Publication (verified):** 2026-11-17T00:11:05Z

**What happened vs when published:**
- Article publication: Nov 17 00:11 UTC.
- Event cited: Axios report on Nov 15. Dateline of release is San Francisco, Nov. 16, 2026.

**Factual summary:** On November 15, 2026 Axios reported that Treasury Secretary Scott Bessent questioned whether NVIDIA’s earnings might have been inflated by inappropriate sales to China. The release says this comes on the heels of a putative securities class action, with Class Period Dec. 19, 2024 – Nov. 12, 2026. The complaint alleges defendants concealed Jensen Huang’s reported personal relationship with an executive at Megaspeed and that assurances including an Oct. 9, 2025 “no evidence of diversion” statement were materially false. Source is a plaintiffs firm: Hagens Berman describes itself as a global plaintiffs’ rights litigation firm.

**Analysis (not fact):** Bearish — regulatory / legal overhang and China-export risk; mixed if allegations remain unproven and no action follows.

### 3. Monday market rout, board stance, export-rental curb
**Headline:** “Nasdaq-100 Drops 2.67% Before Nvidia's $109 Billion Test” — **Markets On Deck via Medium**
**Publication (verified):** 2026-11-17T00:04:49Z, labeled “November 16, 2026 · After Market Tape”

**What happened vs when published:**
- Article publication: Nov 17 00:04 UTC.
- Events described: Monday Nov 16 trading; Tuesday Nov 17 earnings call upcoming.

**Factual summary:** The Nasdaq-100 fell 2.67% Monday while Nvidia closed at $156.14, down 4.12%. The piece cites CNBC reporting, citing a person familiar, that Nvidia’s board was not discussing bringing Jensen Huang back as CEO, and that the Commerce Department moved Monday to close a route that had allowed Chinese firms to rent Nvidia computing power located outside China. Nvidia’s results call is scheduled for Tuesday, November 17, at 5 PM ET, according to the company’s announcement.

**Analysis (not fact):** Bearish — leadership vacuum narrative plus tighter China-access rules pressure sentiment; mixed to bullish if earnings re-anchor on fundamentals.

### 4. Quantum-classical collaboration with IBM and Fermilab
**Headline:** “NVIDIA Announces Collaboration With IBM and Fermi National Accelerator Laboratory to Launch Quantum-Classical Codesign Platform, Plus Error-Correction Breakthrough …” — **NVIDIA via GlobeNewswire, republished by TradingView**
**Publication (verified):** 2026-11-16T16:00:00+00:00

**What happened vs when published:**
- Announcement date and publication date are the same: Chicago, Nov. 16, 2026 — NVIDIA “today announced” plans for collaborations.

**Factual summary:** NVIDIA and IBM announced a quantum supercomputing collaboration combining NVIDIA accelerated computing with IBM quantum computers, with DGX B200 systems to connect with IBM systems including IBM Quantum Nighthawk. NVIDIA also released TCBP-JOBS, part of CUDA-Q, delivering up to 13,000x speedup for error-correction research, cutting computation time for researching new designs from 12 weeks to 45 minutes.

**Analysis (not fact):** Bullish long-term — diversifies technology narrative beyond near-term GPU cycle; limited near-term earnings impact, so mixed for the stock into earnings.

### 5. Q3 FY2027 earnings setup preview
**Headline:** “NVIDIA (NVDA) Set to Release Earnings on Tuesday” — **Defense World**
**Publication (verified date only):** Posted Nov 15th, 2026 — exact time not shown, so placement at window start cannot be precisely verified.

**What happened vs when published:**
- Article publication: Nov 15 (date only).
- Event previewed: Q3 2027 results after close Tuesday, November 17.

**Factual summary:** NVIDIA was projected to post Q3 2027 results after market close Tuesday, November 17th, with analysts expecting $2.47 per share and $108.72 billion revenue, and a call scheduled for Tuesday, November 17, 2026 at 5:00 PM ET.

**Analysis (not fact):** Neutral to mixed — sets a high bar (~$109B) ahead of leadership uncertainty; beat-and-raise would be bullish, miss would be bearish.

**Notes:**
- The Nov 12 resignation of Jensen Huang as president/CEO/director is an older event outside the requested window and is treated here only as context for Nov 15-17 follow-ups.
- No prices, headlines, timestamps or URLs were invented; all details above are attributed to the cited pages.

... (2 outputs left out)

## AMD

I'll search live for AMD's newest market news from the past 48 hours.

I found several candidates in the window — now I'll verify their publication dates and details.

Those checks passed — next I'll confirm the EPYC launch and related items.

As of **2026-11-17 09:29 UTC**, here are 5 distinct AMD developments with verified publication dates, newest first:

### 1. AMD completes World Labs acquisition, premarket slips
**Headline / source:** “AMD Completes World Labs Deal; AMD stock slips 0.64% premarket” — ad-hoc-news.de

**Publication vs event:**
- Article: Published on 11/17/2026 at 08:47 — timezone **not specified** on page
- Event: AMD completed acquisition of World Labs on November 16, as stated on its official website

**Summary:** AMD closed the World Labs deal adding AI research/model expertise to its hardware/software/systems portfolio. The stock traded at 463.25 euros on Lang & Schwarz at 08:44 CET Tuesday, November 17, 2026, 0.64% below the Xetra close Monday at 466.25 euros.

**Implication — analysis, not fact:** Mixed to mildly bullish long-term for AI positioning; mildly bearish short-term if investors focus on integration/dilution and premarket softness.

### 2. Lux supercomputer opens amid $17B quarter test
**Headline / source:** “AMD's $17 Billion Quarter Test: Lux Opens Its Doors While Washington and Nvidia Set the Tone” — ad-hoc-news.de

**Publication vs event:**
- Article: Published: 2026-11-17T07:20:56+01:00 — i.e. 06:20 UTC Nov. 17
- Events discussed: Monday selloff — AMD shed 5.3% on Monday; Q3 reported Nov. 3 — $13.4B revenue, up 45% y/y, non-GAAP EPS $2.09; 6th-gen EPYC availability announced Nov. 10

**Summary:** The article frames a 5.3% Monday drop, citing Reuters on fresh US AI-chip restrictions and Nvidia’s looming earnings, against operational progress. It notes Lux at Oak Ridge opened to scientific users running EPYC, Instinct, Pensando networking and ROCm. Q4 guidance is roughly $17B +/-3%.

**Implication — analysis, not fact:** Mixed: Lux/EPYC breadth supports the data-center execution story, but the high Q4 bar plus sector de-rating risk keeps near-term sentiment bearish until revenue converts.

### 3. Reported 457MW OpenAI-TensorWave contract on AMD Helios
**Headline / source:** “TensorWave says OpenAI deal covers 457MW of AMD-powered…” — Pivot News, by James Okafor

**Publication vs event:**
- Article: James Okafor November 17, 2026 — intraday time/timezone not shown, so date-only
- Underlying event/report date: November 16 report by DatacenterDynamics; future operation dates H1 2027 to early 2028 below

**Summary:** OpenAI contracted TensorWave for 457MW across multiple sites under a six-year agreement, per DCD, powered by AMD Instinct MI455X GPUs in Helios rack-scale systems as dedicated single-tenant capacity. Operations are expected to begin H1 2027 with full deployment by early 2028.

**Implication — analysis, not fact:** Bullish for AMD AI-demand narrative if confirmed, as it implies large-scale MI455X/Helios adoption; execution/delivery timing remains the risk.

### 4. Burry calls AMD “too expensive,” stock down ~14% in 2 sessions
**Headline / source:** “AMD’s Nvidia Rally Fades: Michael Burry Calls AMD ‘Too Expensive As Eternal Alternative’” — Stocktwits

**Publication vs event:**
- Article: Published Nov 16, 2026 | 9:20 PM EST, listed as 2026-11-17T02:20:02Z
- Market event: AMD lost about 14% in last two sessions; closed at $504.28 Monday, up 136% YTD

**Summary:** Burry replied “I agree. I have generally just accepted it is always too expensive as the eternal alternative, first to Intel and then to Nvidia.” He also disclosed holding a large put in VanEck Semiconductor ETF plus puts on Nasdaq-100, Nvidia, MetLife and Palantir. Investors awaited Nvidia’s Tuesday quarterly report as a sector catalyst.

**Implication — analysis, not fact:** Bearish for sentiment/momentum, especially around the $500 support debate; contrarian bullish view would be that washout after a 5-day rally could reset expectations ahead of Nvidia earnings.

### 5. Export-rule selloff hits AMD and chips
**Headline / source:** “Why AMD Stock Was Sliding Today” — The Motley Fool

**Publication vs event:**
- Article: Published: 2026-11-16T16:31:48.000Z
- Event: New Commerce Department rule announced “this morning” restricting chip exports to Southeast Asia; as of 10:53 a.m. ET AMD down 4.4%, VanEck Semiconductor ETF down 4%.

**Summary:** The move followed revelations Singapore’s Megaspeed funnelled Nvidia chips to China against rules. AMD noted prior China-export write-down exposure, with 12.4% of 2025 revenue from Singapore, one of the listed countries. AMD closed at $504.28, -$29.78 (-5.58%), NASDAQ as of 4:00 PM ET 11/16/26.

**Implication — analysis, not fact:** Bearish near-term on fears of blocked/delayed Singapore-region sales and tighter MI350X-class controls; any clarification/narrowing of scope would be the mixed-to-bullish offset to watch.


**Cited sources:**
- [AMD Completes World Labs Deal; AMD stock slips 0.64% premarket](https://www.ad-hoc-news.de/boerse/news/vorboerse/amd-completes-world-labs-deal-amd-stock-slips-0-64-premarket/70525466)
- [AMD's $17 Billion Quarter Test: Lux Opens Its Doors While Washington and Nvidia Set the Tone](https://www.ad-hoc-news.de/boerse/news/unternehmensnachrichten/amd-s-17-billion-quarter-test-lux-opens-its-doors-while-washington-and-nvidia-set-the-tone/70524991)
- [TensorWave says OpenAI deal covers 457MW of AMD-powered… | Pivot News](https://pivotnews.ai/five/tensorwave-says-openai-deal-covers-457mw-of-amd-powered)
- [AMD’s Nvidia Rally Fades: Michael Burry Calls AMD ‘Too Expensive As Eternal Alternative’](https://stocktwits.com/news-articles/markets/equity/amd-s-nvidia-rally-fades-michael-burry-calls-amd-too-expensive-as-eternal-alternative/cNbah8GR9gn)
- [Why AMD Stock Was Sliding Today](https://www.fool.com/investing/2026/11/16/why-amd-stock-was-sliding-today/)

Search calls: 11 | raw pages: 55 | citation annotations: 29


## 6. Inspect raw search results and cited passages

A source appearing in **raw hits** doesn't necessarily mean Muse cited or relied on it. **Citations** show the passages associated with specific source URLs. Neither list guarantees the article's publication date. Open the linked stories to check.

In [ ]:
summary_df = pd.DataFrame(summary_rows)
raw_sources_df = pd.DataFrame(
    source_rows, columns=["ticker", "checked_utc", "title", "url", "snippet"]
).drop_duplicates(subset=["ticker", "url"])
citations_df = pd.DataFrame(
    citation_rows, columns=["ticker", "checked_utc", "title", "url", "supported_passage"]
).drop_duplicates()

print("RAW SEARCH RESULTS (considered sources)")
display(raw_sources_df)
print("\nCITATIONS (sources used in the answer)")
display(citations_df)
print("\nSUMMARY STATUS")
display(summary_df[["ticker", "status"]])

RAW SEARCH RESULTS (considered sources)


,ticker,checked_utc,title,url,snippet
0,META,2026-11-17T09:29:14.737454+00:00,Meta Platforms (META) Faces Fresh Questions As...,https://simplywall.st/stocks/us/media/nasdaq-m...,
1,META,2026-11-17T09:29:14.737454+00:00,Key facts: Meta Platforms Raises $25B for AI; ...,https://www.tradingview.com/news/tradingview:9...,
2,META,2026-11-17T09:29:14.737454+00:00,Meta Platforms (META) Congress trades and insi...,https://aistockwire.com/stock/META,
3,META,2026-11-17T09:29:14.737454+00:00,"Meta Platforms (META) Insider Trades, Congress...",https://alphayou.net/ticker/meta?from=discover,
4,META,2026-11-17T09:29:14.737454+00:00,"Meta Platforms, Inc.",https://trendspider.com/markets/symbols/META/?...,
...,...,...,...,...,...
167,AMD,2026-11-17T09:29:14.737454+00:00,"The Founder's Wire, November 5: Coding Agents ...",https://www.dreaming.press/posts/2026-11-05-fo...,
168,AMD,2026-11-17T09:29:14.737454+00:00,"AI acquisitions and funding in late 2026, and ...",https://primores.org/blog/ai-acquisitions-and-...,
169,AMD,2026-11-17T09:29:14.737454+00:00,"Industry Trends | Maxlin International Co., Ltd.",https://www.maxlin.net/en/trends/,
170,AMD,2026-11-17T09:29:14.737454+00:00,"AI and tech acquisitions, and what happened to...",https://flaviocopes.com/acquisitions/,



CITATIONS (sources used in the answer)


,ticker,checked_utc,title,url,supported_passage
0,META,2026-11-17T09:29:14.737454+00:00,Meta Platforms (META) Faces Fresh Questions As...,https://simplywall.st/stocks/us/media/nasdaq-m...,**Published:** 2026-11-17T02:09:53.834Z — veri...
1,META,2026-11-17T09:29:14.737454+00:00,Meta Platforms (META) Faces Fresh Questions As...,https://simplywall.st/stocks/us/media/nasdaq-m...,**Summary:** Meta is linked to a growing pool ...
2,META,2026-11-17T09:29:14.737454+00:00,Meta Platforms (META) Faces Fresh Questions As...,https://simplywall.st/stocks/us/media/nasdaq-m...,", with large life insurers reported buying com..."
3,META,2026-11-17T09:29:14.737454+00:00,Meta Platforms (META) Faces Fresh Questions As...,https://simplywall.st/stocks/us/media/nasdaq-m...,The structures reportedly rely on Meta guarant...
4,META,2026-11-17T09:29:14.737454+00:00,Meta Platforms (META) Faces Fresh Questions As...,https://simplywall.st/stocks/us/media/nasdaq-m...,", against flagged 2026 AI capex of US$130-145B..."
...,...,...,...,...,...
70,AMD,2026-11-17T09:29:14.737454+00:00,Why AMD Stock Was Sliding Today,https://www.fool.com/investing/2026/11/16/why-...,- Event: New Commerce Department rule announce...
71,AMD,2026-11-17T09:29:14.737454+00:00,Why AMD Stock Was Sliding Today,https://www.fool.com/investing/2026/11/16/why-...,"; as of 10:53 a.m. ET AMD down 4.4%, VanEck Se..."
72,AMD,2026-11-17T09:29:14.737454+00:00,Why AMD Stock Was Sliding Today,https://www.fool.com/investing/2026/11/16/why-...,**Summary:** The move followed revelations Sin...
73,AMD,2026-11-17T09:29:14.737454+00:00,Why AMD Stock Was Sliding Today,https://www.fool.com/investing/2026/11/16/why-...,AMD noted prior China-export write-down exposu...



SUMMARY STATUS


,ticker,status
0,META,Grounded response (verify article timestamps)
1,NVDA,Grounded response (verify article timestamps)
2,AMD,Grounded response (verify article timestamps)


## 7. Save as CSV and Markdown

Files appear in the Colab **Files** sidebar. Optionally download them with `google.colab.files.download(...)`.

In [ ]:
raw_sources_df.to_csv("stock_news_raw_sources.csv", index=False)
citations_df.to_csv("stock_news_citations.csv", index=False)
summary_df.to_csv("stock_news_summaries.csv", index=False)

with open("stock_news_report.md", "w", encoding="utf-8") as f:
    f.write(f"# Stock news scan — {checked_at:%Y-%m-%d %H:%M} UTC\n\n")
    f.write(f"Requested window: {window_start:%Y-%m-%d %H:%M} UTC to {checked_at:%Y-%m-%d %H:%M} UTC\n\n")
    for record in summary_rows:
        f.write(f"## {record['ticker']}\n\n**Status:** {record['status']}\n\n")
        if record.get("answer"):
            f.write(record["answer"] + "\n\n")
            urls = dict.fromkeys(
                c["url"] for c in citation_rows if c["ticker"] == record["ticker"]
            )
            if urls:
                f.write("**Cited URLs:**\n\n")
                f.write("\n".join(f"- {url}" for url in urls) + "\n\n")

print("Saved stock_news_report.md and three CSV files.")
# Optional Colab download:
# from google.colab import files
# files.download("stock_news_report.md")

Saved stock_news_report.md and three CSV files.


---

### Limitations to keep in mind

- **Not a real-time trading feed:** Search indexing and API latency are unknown and not suitable for reacting to news in seconds.
- **Date filtering is a prompt request, not guaranteed enforcement.** The notebook does not independently validate timestamps or completeness.
- **Market implications are hypotheses, not buy/sell recommendations.** Cross-check primary sources and price moves before using in a strategy.
- **Cost:** You pay for model usage plus any web searches; one ticker request can involve multiple search calls.

Next possible extension: combine a dedicated timestamped financial news feed with Muse for classification and an IBKR data feed for event-study analysis. That avoids relying on search-grounding as a guaranteed news discovery engine.